# Benchmark: Pandas vs. Polars vs. PySpark

1. **Load CSV** (`raw_data.csv`)
2. **Compute discount columns** (`discount_pct`, `is_on_sale`)
3. **Filter rows** where `compare_price < current_price`
4. **Sort + drop duplicates by `sku`** (keep newest by `created_at`)

The final cell prints a comparison table in the same shape as the reference: one block per
operation with four metric rows and three library columns.


## 0. Install dependencies


In [2]:
%pip install pandas polars pyspark psutil


Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install jinja2

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and constants


In [ ]:
import os
import time
import gc
import tracemalloc

import psutil
import pandas as pd
import polars as pl

INPUT_FILE = r"C:\Users\Owner\web_scrape\raw_data.csv"

# Try to start PySpark. If Java is missing skip its benchmarks.
SPARK_AVAILABLE = False
spark = None
try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    spark = (
        SparkSession.builder
        .master("local[*]")
        .appName("clean_data_v2_benchmark")
        .config("spark.driver.memory", "2g")
        .config("spark.sql.shuffle.partitions", "4")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    SPARK_AVAILABLE = True
    print("PySpark ready.")
except Exception as e:
    print("PySpark could not start; its column will be skipped.")
    print("Reason:", e)


## 2. Benchmark helper

`benchmark(func)` runs `func()` once and records:

- **time_s**  -- wall-clock seconds (`time.perf_counter`)
- **peak_mem_mb** -- peak Python-allocated memory during the run (`tracemalloc`)
- **cpu_pct** -- CPU usage of this process while the function ran (`psutil`)
- **throughput** -- rows / second (we pass `n_rows` in afterwards)

Note: for PySpark the heavy memory lives in the JVM, which `tracemalloc` doesn't see -- so
its peak-memory number will look very small (same caveat as the reference table).


In [5]:
def benchmark(func):
    """Run func() once, return metrics dict and the returned result."""
    gc.collect()  # start from a clean slate

    process = psutil.Process(os.getpid())
    tracemalloc.start()
    process.cpu_percent(interval=None)  # initialise CPU counter

    t0 = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - t0

    cpu_pct = process.cpu_percent(interval=None)
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    metrics = {
        "time_s": elapsed,
        "peak_mem_mb": peak_bytes / (1024 * 1024),
        "cpu_pct": cpu_pct,
    }
    return metrics, result


def run_three(operation_name, pandas_fn, polars_fn, pyspark_fn, n_rows, results):
    """Run the same operation in Pandas / Polars / PySpark and append metrics to results."""
    print(f"\n--- {operation_name} ---")

    m_pd, _ = benchmark(pandas_fn)
    m_pd["throughput"] = n_rows / m_pd["time_s"] if m_pd["time_s"] > 0 else 0
    m_pd["operation"] = operation_name
    m_pd["library"]   = "Pandas"
    results.append(m_pd)
    print(f"Pandas : {m_pd['time_s']:.4f}s  mem={m_pd['peak_mem_mb']:.4f}MB  cpu={m_pd['cpu_pct']:.1f}%  thr={m_pd['throughput']:.2f}")

    m_pl, _ = benchmark(polars_fn)
    m_pl["throughput"] = n_rows / m_pl["time_s"] if m_pl["time_s"] > 0 else 0
    m_pl["operation"] = operation_name
    m_pl["library"]   = "Polars"
    results.append(m_pl)
    print(f"Polars : {m_pl['time_s']:.4f}s  mem={m_pl['peak_mem_mb']:.4f}MB  cpu={m_pl['cpu_pct']:.1f}%  thr={m_pl['throughput']:.2f}")

    if SPARK_AVAILABLE and pyspark_fn is not None:
        m_sk, _ = benchmark(pyspark_fn)
        m_sk["throughput"] = n_rows / m_sk["time_s"] if m_sk["time_s"] > 0 else 0
        m_sk["operation"] = operation_name
        m_sk["library"]   = "PySpark"
        results.append(m_sk)
        print(f"PySpark: {m_sk['time_s']:.4f}s  mem={m_sk['peak_mem_mb']:.4f}MB  cpu={m_sk['cpu_pct']:.1f}%  thr={m_sk['throughput']:.2f}")


## 3. Operation 1 -- Load CSV

From v2: `pd.read_csv(INPUT_FILE, encoding="utf-8-sig")`.


In [6]:
results = []  # collects metrics across all operations


def op1_pandas():
    # Force Pandas to read 'SKU' as a string (object)
    return pd.read_csv(INPUT_FILE, encoding="utf-8-sig", dtype={"SKU": str})


def op1_polars():
    # Force Polars to read 'SKU' as a String to fix the ComputeError
    return pl.read_csv(INPUT_FILE, schema_overrides={"SKU": pl.String})


def op1_pyspark():
    # PySpark can inferSchema, but forcing the schema via a DDL string 
    # or updating the inferred type ensures it treats SKU as a string safely.
    # Alternatively, you can use `.option("inferSchema", "true")` but PySpark 
    # usually falls back to string automatically if it encounters letters. 
    # To be perfectly fair to the others, we let it infer or explicitly cast if needed:
    df = spark.read.csv(INPUT_FILE, header=True, inferSchema=True, multiLine=True, escape='"')
    df.count()   # force the lazy plan to materialise
    return df


# Use the pandas row count as the canonical N for throughput
_n_rows_holder = {}
def op1_pandas_with_count():
    df = op1_pandas()
    _n_rows_holder["n"] = len(df)
    return df

m_pd, df_pd = benchmark(op1_pandas_with_count)
N_ROWS = _n_rows_holder["n"]
m_pd["throughput"] = N_ROWS / m_pd["time_s"]
m_pd["operation"] = "Load CSV"
m_pd["library"]   = "Pandas"
results.append(m_pd)
print(f"Pandas : {m_pd['time_s']:.4f}s  mem={m_pd['peak_mem_mb']:.4f}MB  cpu={m_pd['cpu_pct']:.1f}%  thr={m_pd['throughput']:.2f}   (N={N_ROWS:,})")

m_pl, _ = benchmark(op1_polars)
m_pl["throughput"] = N_ROWS / m_pl["time_s"]
m_pl["operation"] = "Load CSV"
m_pl["library"]   = "Polars"
results.append(m_pl)
print(f"Polars : {m_pl['time_s']:.4f}s  mem={m_pl['peak_mem_mb']:.4f}MB  cpu={m_pl['cpu_pct']:.1f}%  thr={m_pl['throughput']:.2f}")

if SPARK_AVAILABLE:
    m_sk, _ = benchmark(op1_pyspark)
    m_sk["throughput"] = N_ROWS / m_sk["time_s"]
    m_sk["operation"] = "Load CSV"
    m_sk["library"]   = "PySpark"
    results.append(m_sk)
    print(f"PySpark: {m_sk['time_s']:.4f}s  mem={m_sk['peak_mem_mb']:.4f}MB  cpu={m_sk['cpu_pct']:.1f}%  thr={m_sk['throughput']:.2f}")

Pandas : 2.1964s  mem=237.1638MB  cpu=100.3%  thr=70438.78   (N=154,714)
Polars : 0.0761s  mem=0.0087MB  cpu=369.4%  thr=2033321.37
PySpark: 3.7284s  mem=0.8371MB  cpu=0.4%  thr=41495.66


## Preload the data for the remaining operations

The next three operations all start from a loaded DataFrame, so we load once outside the
timed sections.


In [7]:
pdf = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
ldf = pl.read_csv(INPUT_FILE, schema_overrides={"SKU": pl.String})

if SPARK_AVAILABLE:
    sdf = spark.read.csv(INPUT_FILE, header=True, inferSchema=True, multiLine=True, escape='"').cache()
    sdf.count()  # materialise & cache so the cache hit (not a re-read) is what the benchmarks measure
else:
    sdf = None

print(f"Pandas rows : {len(pdf):,}")
print(f"Polars rows : {len(ldf):,}")
if SPARK_AVAILABLE:
    print(f"PySpark rows: {sdf.count():,}")


Pandas rows : 154,714
Polars rows : 154,714
PySpark rows: 154,714


## 4. Operation 2 -- Compute `discount_pct` + `is_on_sale`

From v2:
```python
df["discount_pct"] = (df["compare_price"] - df["current_price"]) / df["compare_price"] * 100
df["is_on_sale"]   = df["discount_pct"] > 5
```


In [8]:
def op2_pandas():
    df = pdf.copy()
    df["discount_pct"] = (df["Compare at Price"] - df["Current Price"]) / df["Compare at Price"] * 100
    df["is_on_sale"]   = df["discount_pct"] > 5
    return df


def op2_polars():
    return (
        ldf
        .with_columns(
            ((pl.col("Compare at Price") - pl.col("Current Price")) / pl.col("Compare at Price") * 100).alias("discount_pct")
        )
        .with_columns(
            (pl.col("discount_pct") > 5).alias("is_on_sale")
        )
    )


def op2_pyspark():
    df = sdf.withColumn(
        "discount_pct",
        (F.col("Compare at Price") - F.col("Current Price")) / F.col("Compare at Price") * 100,
    )
    df = df.withColumn("is_on_sale", F.col("discount_pct") > 5)
    df.count()  # force materialisation
    return df


run_three(
    "Compute discount columns",
    op2_pandas,
    op2_polars,
    op2_pyspark if SPARK_AVAILABLE else None,
    N_ROWS,
    results,
)



--- Compute discount columns ---
Pandas : 0.0177s  mem=22.7274MB  cpu=176.2%  thr=8746141.76
Polars : 0.0086s  mem=0.1617MB  cpu=1451.9%  thr=18084417.48
PySpark: 0.1475s  mem=0.1666MB  cpu=0.0%  thr=1049020.84


## 5. Operation 3 -- Filter rows where `compare_price < current_price`

From v2:
```python
bad_rows = df["compare_price"] < df["current_price"]
df = df[~bad_rows]
```


In [9]:
def op3_pandas():
    bad = pdf["Compare at Price"] < pdf["Current Price"]
    return pdf[~bad]


def op3_polars():
    return ldf.filter(~(pl.col("Compare at Price") < pl.col("Current Price")))


def op3_pyspark():
    out = sdf.filter(~(F.col("Compare at Price") < F.col("Current Price")))
    out.count()
    return out


run_three(
    "Filter negative-discount rows",
    op3_pandas,
    op3_polars,
    op3_pyspark if SPARK_AVAILABLE else None,
    N_ROWS,
    results,
)



--- Filter negative-discount rows ---
Pandas : 0.0189s  mem=17.9737MB  cpu=82.7%  thr=8201504.44
Polars : 0.0057s  mem=0.0036MB  cpu=0.0%  thr=26999755.68
PySpark: 0.1898s  mem=0.2971MB  cpu=0.0%  thr=815176.61


## 6. Operation 4 -- Sort + dedup by `SKU` (keep newest by `Created At`)

From v2 (first dedup pass):
```python
df = df.sort_values("created_at", kind="mergesort")
df = df.drop_duplicates(subset="sku", keep="last")
```

Polars: `sort(...).unique(subset="SKU", keep="last")`.
PySpark: window-function pattern (`row_number()` over `partitionBy("SKU").orderBy("Created At" desc)`).


In [10]:
def op4_pandas():
    out = pdf.sort_values("Created At", kind="mergesort")
    out = out.drop_duplicates(subset="SKU", keep="last")
    return out


def op4_polars():
    return ldf.sort("Created At").unique(subset="SKU", keep="last")


def op4_pyspark():
    w = Window.partitionBy("SKU").orderBy(F.col("Created At").desc())
    out = (
        sdf
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )
    out.count()
    return out


run_three(
    "Sort + drop duplicates by SKU",
    op4_pandas,
    op4_polars,
    op4_pyspark if SPARK_AVAILABLE else None,
    N_ROWS,
    results,
)



--- Sort + drop duplicates by SKU ---
Pandas : 0.0763s  mem=34.3865MB  cpu=102.3%  thr=2027889.81
Polars : 0.0245s  mem=0.0034MB  cpu=1020.5%  thr=6327253.39
PySpark: 1.0748s  mem=0.3393MB  cpu=2.9%  thr=143948.54


## 7. Final comparison table

Same shape as the reference table: one block per operation, four metric rows per block,
one column per library.


In [11]:
res_df = pd.DataFrame(results)

# Long -> wide so each metric is a row and each library is a column
metric_labels = {
    "time_s":      "Code Execution Time (s)",
    "peak_mem_mb": "Peak Memory Usage (MB)",
    "cpu_pct":     "CPU Usage (%)",
    "throughput":  "Throughput (rows/s)",
}

long = res_df.melt(
    id_vars=["operation", "library"],
    value_vars=list(metric_labels.keys()),
    var_name="metric",
    value_name="value",
)
long["metric"] = long["metric"].map(metric_labels)

library_order = ["Pandas", "Polars", "PySpark"]
library_order = [c for c in library_order if c in long["library"].unique()]

table = (
    long
    .pivot_table(index=["operation", "metric"], columns="library", values="value", sort=False)
    .reindex(columns=library_order)
)

# Keep the metric rows in the same order as the reference table
metric_order = list(metric_labels.values())
table = table.reindex(metric_order, level="metric")

# Round the underlying data
table = table.round(4)

# --- ADD THE FIX HERE ---
# This formats everything cleanly as decimals and adds commas for big numbers
table_styled = table.style.format(lambda x: f"{x:,.4f}" if x > 1 else f"{x:.4f}")

# Call this at the end to display it in your notebook
table_styled

## 8. Notes / caveats

- **Peak memory** is measured with `tracemalloc`, which only sees Python-allocated memory.
  Polars stores most of its data in Arrow buffers and PySpark stores its data in the JVM --
  neither of those is visible to `tracemalloc`, so those numbers look very small. This
  matches the behaviour of the reference table.
- **CPU usage** is the host process's CPU share during the timed window. Polars uses all
  available cores by default, so its number can hit ~100% even for short operations.
- **Single run.** For a more robust comparison, wrap each `benchmark(...)` call in a small
  loop and take the median. One run is enough for ballpark numbers.
- **PySpark first call is slow.** The first Spark operation in a session pays JVM warm-up
  and Catalyst initialisation costs. If you want a fairer Spark number, run the notebook
  twice and use the second-run figures.
